# AI-powered 6G RAN Optimization - Colab Notebook

This notebook generates synthetic RAN data, trains models, runs inference, and visualizes outcomes.

**Run the Setup cell first.** It clones the repository (in Colab), makes the project packages importable, and installs dependencies. If the clone fails it raises a clear error instead of failing later with `ModuleNotFoundError`.

In [ ]:
# === Setup (run this first) ==========================================
# Makes the project importable in Google Colab or locally, then installs deps.
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/erdioz/AI-powered-6G-RAN-Optimization-System.git"
REPO_DIR = "AI-powered-6G-RAN-Optimization-System"


def is_project_root(path="."):
    return (os.path.isfile(os.path.join(path, "pyproject.toml"))
            and os.path.isfile(os.path.join(path, "data", "generator.py")))


# 1) Make sure we are inside a populated copy of the project.
if not is_project_root("."):
    # Clear away an empty/partial clone left by a previous failed attempt.
    if os.path.isdir(REPO_DIR) and not is_project_root(REPO_DIR):
        shutil.rmtree(REPO_DIR, ignore_errors=True)
    if not os.path.isdir(REPO_DIR):
        print("Cloning", REPO_URL)
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    os.chdir(REPO_DIR)

# 2) Fail loudly with a clear message if the project still isn't here.
if not is_project_root("."):
    raise RuntimeError(
        f"Project files not found in {os.getcwd()!r}. The clone may have failed - "
        "re-run this cell, or check network access to GitHub."
    )

# 3) Put the project root on sys.path so `import data`, `import pipeline`, ... work.
ROOT = os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

# 4) Install dependencies (editable install also registers the `ran6g` CLI).
if subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."]).returncode != 0:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

print("Setup complete.")
print("  project root :", ROOT)
print("  data present :", os.path.isfile("data/generator.py"))

In [ ]:
from data.generator import SyntheticRANDataGenerator
from pipeline.trainer import train_all
from pipeline.inference import RANInferenceService
from visualization.plots import plot_ue_movement, plot_sinr_over_time, plot_beam_selection, plot_anomalies
import pandas as pd
from IPython.display import Image, display

In [ ]:
generator = SyntheticRANDataGenerator()
df = generator.generate()
df.to_csv('data/sample_dataset.csv', index=False)
df.head()

In [ ]:
metrics = train_all('data/sample_dataset.csv')
metrics

In [ ]:
service = RANInferenceService()
sample = df.iloc[0].to_dict()

# Build each payload from the model's own feature list so it stays in sync with the models.
qos_payload = {k: sample[k] for k in service.qos.config.feature_columns}
beam_payload = {k: sample[k] for k in service.beam.config.feature_columns}
anom_payload = {k: sample[k] for k in service.anomaly.config.feature_columns}

print(service.predict_qos(qos_payload))
print(service.select_beam(beam_payload))
print(service.detect_anomaly(anom_payload))

In [ ]:
plot_ue_movement(df, user_id=0)
plot_sinr_over_time(df, user_id=0)
plot_beam_selection(df, user_id=0)
plot_anomalies(df)
display(Image('outputs/plots/ue_movement.png'))
display(Image('outputs/plots/sinr_over_time.png'))
display(Image('outputs/plots/beam_selection.png'))
display(Image('outputs/plots/anomalies.png'))